# Comparative Specifications — DiD Analysis

**Standalone notebook** — loads pre-processed data from `analysis_data.parquet`.
Run `preprocess.py` once to generate the parquet from the two Excel source files.

## Outline
| Section | Specification | Key coefficient |
|---|---|---|
| 0 | Data loading & sample construction | — |
| 1 | Simple OLS | `treat` |
| 2 | Difference-in-Differences | `treat:post` |
| 3 | Triple Difference (DDD) | `treat:post:impl` |
| 4 | DDD — high-dimensional FEs | `treat:post:impl` |
| 5 | Comparison table | all specs side-by-side |
| 6 | Doubly-Robust DDD | `ATT_dr` |

## 0. Data Loading & Feature Engineering

### 0.1 Imports

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col

BASE = Path('.').resolve()  # folder containing the data files
print(f"Working directory: {BASE}")

Working directory: C:\Users\zolb\Files C\ClaudeFolder\EcoLink


### 0.2 Load data

In [10]:
df = pd.read_parquet(BASE / 'analysis_data.parquet')
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'  years:      {sorted(df["year"].unique())}')
print(f'  countries:  {df["ipscntry"].nunique()}')
print(f'  treat=1:    {(df["treat"]==1).sum():,}  |  treat=0: {(df["treat"]==0).sum():,}')
print(f'  impl=1 cntrs: {df.loc[df["implementation_country"]==1,"ipscntry"].nunique()}')
print(f'  impl=0 cntrs: {df.loc[df["implementation_country"]==0,"ipscntry"].nunique()}')

Loaded: 28,263 rows x 42 cols
  years:      [np.int64(2021), np.int64(2024)]
  countries:  27
  treat=1:    4,236  |  treat=0: 24,000
  impl=1 cntrs: 20
  impl=0 cntrs: 7


### 0.3 Treatment variables
`treat = 1` if firm has ≥ 250 employees **or** turnover ≥ €10 M.
Pre-computed by `preprocess.py`; confirmed below.

In [11]:
# Treatment variables loaded from parquet — verify key columns
print('treat distribution:')
print(df['treat'].value_counts(dropna=False).to_string())
print('post distribution:')
print(df['post'].value_counts().to_string())

treat distribution:
treat
0.0    24000
1.0     4236
NaN       27
post distribution:
post
0    14215
1    14048


### 0.4 Outcome variables
Pre-computed by `preprocess.py`.

- **`target_engaged`**: 1 if firm has or plans a climate strategy (from q14)
- **`firm_growth_ord`**: ordinal −1/0/1 (decreased / unchanged / increased)
- **`firm_growth_ord_d`**: binary 1 = increased, 0 = flat or declined

In [12]:
# Outcome variables loaded from parquet — verify means
for col in ['target_engaged', 'firm_growth_ord', 'firm_growth_ord_d']:
    print(f'  {col}: mean={df[col].mean():.3f}  N={df[col].notna().sum():,}')

  target_engaged: mean=0.508  N=28,263
  firm_growth_ord: mean=0.231  N=28,263
  firm_growth_ord_d: mean=0.420  N=28,263


### 0.5 Firm-level controls
Pre-computed by `preprocess.py`. Available columns:

| Variable | Description |
|---|---|
| `firm_age_ord` | Ordinal 1–4 (1=youngest) |
| `firm_age_ord_d` | Binary: 1 if founded before 2016/2014 |
| `east_west` | 1=Western EU, 0=Eastern EU |
| `*_maturity_d` | 10 resource-efficiency domain action dummies |
| `green_staff_any` | 1 if any green-dedicated staff |
| `investment_dummy` | 1 if any green investment |
| `barrier_*` / `barrier_*_d` | Institutional / capability / market barriers |
| `support_*` | Knowledge, financial, external, internal support |
| `green_offer_ord_d` / `green_turnover_pct_d` | Green product/revenue dummies |

In [13]:
# Controls loaded from parquet — spot-check
ctrl_cols = ['firm_age_ord','east_west','green_staff_any','investment_dummy',
             'barrier_institutional_d','support_knowledge_d','green_offer_ord_d']
print(df[ctrl_cols].describe().round(3).to_string())

       firm_age_ord  east_west  green_staff_any  investment_dummy  barrier_institutional_d  support_knowledge_d  green_offer_ord_d
count     28263.000  28263.000        28263.000         28263.000                28263.000            28263.000          28263.000
mean          3.695      0.580            0.486             0.825                    0.492                0.596              0.466
std           0.688      0.493            0.500             0.380                    0.500                0.491              0.499
min           1.000      0.000            0.000             0.000                    0.000                0.000              0.000
25%           4.000      0.000            0.000             1.000                    0.000                0.000              0.000
50%           4.000      1.000            0.000             1.000                    0.000                1.000              0.000
75%           4.000      1.000            1.000             1.000                  

### 0.6 Analysis samples

In [14]:
# Main sample: implementing countries (impl=1), complete cases
df_main  = df[df['sample_main'] == 1]
df_clean = df_main.dropna(subset=[
    "target_engaged", "treat", "post", "firm_age_ord", "nace_b", "ipscntry"
]).copy()

# DDD sample: all countries (impl=0 + impl=1)
df_ddd = df.dropna(subset=[
    "target_engaged", "treat", "post", "firm_age_ord", "nace_b",
    "ipscntry", "implementation_country"
]).copy()
df_ddd["impl"] = df_ddd["implementation_country"]

# Working samples (drop residual NAs on firm_growth_ord_d)
df_s = df_clean.dropna(subset=['firm_growth_ord_d']).copy()  # OLS / DiD
df_d = df_ddd.dropna(subset=['firm_growth_ord_d']).copy()    # DDD

# Extended FE dummies for Section 4
df_d["cntry_treat"] = df_d["ipscntry"].astype(str) + "_" + df_d["treat"].astype(str)
df_d["cntry_post"]  = df_d["ipscntry"].astype(str) + "_" + df_d["post"].astype(str)

print(f"OLS/DiD sample (impl=1 only): {df_s.shape[0]:,} obs")
print(f"DDD sample (all countries):   {df_d.shape[0]:,} obs")
print(f"  impl=1 (policy countries):  {(df_d['impl']==1).sum():,}")
print(f"  impl=0 (never-treated):     {(df_d['impl']==0).sum():,}")

def cl(df_):
    """Clustered SE by country."""
    return {"cov_type": "cluster", "cov_kwds": {"groups": df_["ipscntry"]}}

OLS/DiD sample (impl=1 only): 21,396 obs
DDD sample (all countries):   28,236 obs
  impl=1 (policy countries):  21,396
  impl=0 (never-treated):     6,840


---
## 1. Simple OLS
Cross-sectional estimate of the firm-size (`treat`) effect with no pre/post
variation exploited. Two outcomes × two FE choices = 4 models.

| Model | Outcome | Country FE |
|---|---|---|
| 1a | `target_engaged` | No |
| 1b | `target_engaged` | Yes (`C(ipscntry)`) |
| 1c | `firm_growth_ord_d` | No |
| 1d | `firm_growth_ord_d` | Yes (`C(ipscntry)`) |

In [15]:
s1a = smf.ols("target_engaged    ~ treat + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s1b = smf.ols("target_engaged    ~ treat + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))
s1c = smf.ols("firm_growth_ord_d ~ treat + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s1d = smf.ols("firm_growth_ord_d ~ treat + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))

for lbl, m in [("1a  target_engaged    no FE",      s1a),
               ("1b  target_engaged    country FE",  s1b),
               ("1c  firm_growth_ord_d no FE",       s1c),
               ("1d  firm_growth_ord_d country FE",  s1d)]:
    print(f"{lbl}: treat = {m.params['treat']:.4f}  (p={m.pvalues['treat']:.3f})")

1a  target_engaged    no FE: treat = 0.2608  (p=0.000)
1b  target_engaged    country FE: treat = 0.2302  (p=0.000)
1c  firm_growth_ord_d no FE: treat = 0.1696  (p=0.000)
1d  firm_growth_ord_d country FE: treat = 0.1626  (p=0.000)


---
## 2. Difference-in-Differences (DiD)
`treat:post` is the DiD coefficient — the additional change for large firms
after the policy (`post=1`, year 2024) relative to small firms.  
Sample restricted to implementing countries (`impl=1`).

| Model | Outcome | Country FE |
|---|---|---|
| 2a | `target_engaged` | No |
| 2b | `target_engaged` | Yes |
| 2c | `firm_growth_ord_d` | No |
| 2d | `firm_growth_ord_d` | Yes |

In [16]:
s2a = smf.ols("target_engaged    ~ treat*post + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s2b = smf.ols("target_engaged    ~ treat*post + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))
s2c = smf.ols("firm_growth_ord_d ~ treat*post + C(nace_b) + C(firm_age_ord)",                data=df_s).fit(**cl(df_s))
s2d = smf.ols("firm_growth_ord_d ~ treat*post + C(nace_b) + C(firm_age_ord) + C(ipscntry)", data=df_s).fit(**cl(df_s))

for lbl, m in [("2a  target_engaged    no FE",      s2a),
               ("2b  target_engaged    country FE",  s2b),
               ("2c  firm_growth_ord_d no FE",       s2c),
               ("2d  firm_growth_ord_d country FE",  s2d)]:
    print(f"{lbl}: treat:post = {m.params['treat:post']:.4f}  (p={m.pvalues['treat:post']:.3f})")

2a  target_engaged    no FE: treat:post = 0.0634  (p=0.028)
2b  target_engaged    country FE: treat:post = 0.0649  (p=0.014)
2c  firm_growth_ord_d no FE: treat:post = -0.1062  (p=0.005)
2d  firm_growth_ord_d country FE: treat:post = -0.1004  (p=0.007)


---
## 3. Difference-in-Difference-in-Differences (DDD)
Adds `impl` (= `implementation_country`) as a third dimension.  
Countries with `impl=0` are **never-treated controls**, removing global time
trends that affect large firms everywhere regardless of policy.  
`treat:post:impl` is the DDD coefficient.

> **Note:** `C(ipscntry)` cannot be included — `impl` is constant within
> countries and is perfectly collinear with country FEs. Sector + age FEs
> are used for the 'with FE' variant.

| Model | Outcome | Controls |
|---|---|---|
| 3a | `target_engaged` | Sector only |
| 3b | `target_engaged` | Sector + firm age |
| 3c | `firm_growth_ord_d` | Sector only |
| 3d | `firm_growth_ord_d` | Sector + firm age |

In [17]:
s3a = smf.ols("target_engaged    ~ treat*post*impl + C(nace_b)",                   data=df_d).fit(**cl(df_d))
s3b = smf.ols("target_engaged    ~ treat*post*impl + C(nace_b) + C(firm_age_ord)", data=df_d).fit(**cl(df_d))
s3c = smf.ols("firm_growth_ord_d ~ treat*post*impl + C(nace_b)",                   data=df_d).fit(**cl(df_d))
s3d = smf.ols("firm_growth_ord_d ~ treat*post*impl + C(nace_b) + C(firm_age_ord)", data=df_d).fit(**cl(df_d))

for lbl, m in [("3a  target_engaged    sector FE",      s3a),
               ("3b  target_engaged    sector+age FE",  s3b),
               ("3c  firm_growth_ord_d sector FE",      s3c),
               ("3d  firm_growth_ord_d sector+age FE",  s3d)]:
    print(f"{lbl}: treat:post:impl = {m.params['treat:post:impl']:.4f}  "
          f"(p={m.pvalues['treat:post:impl']:.3f})")

3a  target_engaged    sector FE: treat:post:impl = 0.1749  (p=0.001)
3b  target_engaged    sector+age FE: treat:post:impl = 0.1754  (p=0.001)
3c  firm_growth_ord_d sector FE: treat:post:impl = -0.0520  (p=0.427)
3d  firm_growth_ord_d sector+age FE: treat:post:impl = -0.0524  (p=0.430)


---
## 4. Doubly-Robust DDD

Implements the **DR DDD estimator** from Ortiz-Villavicencio & Sant'Anna (2025),
"Better Understanding Triple Differences Estimators" (OVS 2025), eq. 4.1.

### Why we need it

Section 3 uses the three-way fixed effects (3WFE) regression
`Y ~ treat*post*impl + FEs`.  OVS (2025) show this estimator is **generally biased** when
covariates are needed to justify the conditional DDD parallel-trends assumption (DDD-CPT).
The bias arises because 3WFE integrates covariates over the *control* distribution instead
of the *treated* distribution (paper sec. 3.1, Figure 1).

### Identification structure

In our setting a firm is **effectively treated** if it satisfies **two** criteria:

| Paper symbol | Our variable | Meaning |
|---|---|---|
| Q = 1 (eligible) | `treat = 1` | Large firm above CSRD threshold |
| S = g (enabling) | `impl = 1` | Country transposed the directive |

The never-enabling countries (`impl = 0`) serve as the comparison group (`g_c = inf`).
Standard errors are clustered by country (`ipscntry`, 27 clusters).

### Estimator

The DR DDD decomposes into **three DR DiD components** (OVS 2025, eq. 4.1):

$$\widehat{ATT}_{\text{dr-ddd}} = \widehat{ATT}_{k3} + \widehat{ATT}_{k2} - \widehat{ATT}_{k1}$$

| Component | Treated group | Comparison group | Identifies |
|---|---|---|---|
| $k_3$ | impl=1, treat=1 | impl=1, treat=0 | Within-country eligibility DiD |
| $k_2$ | impl=1, treat=1 | impl=0, treat=1 | Between-country, among eligible |
| $k_1$ | impl=1, treat=1 | impl=0, treat=0 | Between-country, between eligibility |

Each component uses the **Sant'Anna & Zhao (2020) DR DiD** for repeated cross-sections:
a logistic propensity score and linear outcome regressions for each (group × period) cell.

The ATT is doubly robust: consistent if **either** the propensity score model **or**
the outcome regression model is correctly specified (not necessarily both).

Standard errors are computed from the combined **influence function**, clustered by country.

### 4.1  Covariates and DR estimation setup

In [21]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from scipy import stats as scipy_stats
import warnings

# One-hot sector FEs + firm age (ordinal numeric) + east/west
sector_dummies = pd.get_dummies(df_ddd['nace_b'], prefix='sec', drop_first=True)
X_cols_dr = sector_dummies.columns.tolist() + ['firm_age_ord', 'east_west']

df_dr_base = pd.concat([df_ddd.reset_index(drop=True),
                         sector_dummies.reset_index(drop=True)], axis=1)

df_dr = df_dr_base.dropna(subset=X_cols_dr + ['implementation_country']).copy()
print(f'DR sample: {len(df_dr):,} obs, {df_dr["ipscntry"].nunique()} countries')
print(f'impl=1 countries: {df_dr.loc[df_dr["implementation_country"]==1,"ipscntry"].nunique()}')
print(f'impl=0 countries: {df_dr.loc[df_dr["implementation_country"]==0,"ipscntry"].nunique()}')

for name, cond in [('impl=1 treat=1', (df_dr['implementation_country']==1)&(df_dr['treat']==1)),
                   ('impl=1 treat=0', (df_dr['implementation_country']==1)&(df_dr['treat']==0)),
                   ('impl=0 treat=1', (df_dr['implementation_country']==0)&(df_dr['treat']==1)),
                   ('impl=0 treat=0', (df_dr['implementation_country']==0)&(df_dr['treat']==0))]:
    print(f'  {name}: {cond.sum():,}')

DR sample: 28,236 obs, 27 countries
impl_cyp=1 countries: 19
impl_cyp=0 countries: 8
  impl=1 treat=1: 3,199
  impl=1 treat=0: 17,693
  impl=0 treat=1: 1,037
  impl=0 treat=0: 6,307


### 4.2  DR DiD building block (repeated cross-sections)

In [22]:
def dr_did_rc(Y, post, treated, X_mat, cluster):
    """
    Doubly-robust DiD for repeated cross-sections.
    Sant'Anna & Zhao (2020), adapted as a building block for the DR DDD.

    Parameters
    ----------
    Y       : outcome (1-D array, length n_k)
    post    : 1=post period, 0=pre (1-D array, length n_k)
    treated : 1=treated group, 0=comparison group (1-D array, length n_k)
    X_mat   : covariate matrix WITHOUT intercept (n_k x p)
    cluster : cluster labels for SE (1-D array, length n_k)

    Returns
    -------
    att : float  -- DR DiD point estimate
    psi : array  -- influence function (length n_k), mean = att
    """
    n_k = len(Y)

    # 1. Propensity score: P(treated=1 | X) via logistic regression
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        lr = LogisticRegression(max_iter=2000, solver='lbfgs', C=1e6)
        lr.fit(X_mat, treated)
    ps = lr.predict_proba(X_mat)[:, 1]
    ps = np.clip(ps, 1e-6, 1 - 1e-6)

    # 2. Outcome regressions: E[Y | comparison_group, period, X]
    for prd in [1, 0]:
        mask = (treated == 0) & (post == prd)
        if mask.sum() < 5:
            raise ValueError(f'Too few comparison obs in period {prd}: {mask.sum()}')
        coef = LinearRegression().fit(X_mat[mask], Y[mask])
        if prd == 1:
            m_post = coef.predict(X_mat)
        else:
            m_pre  = coef.predict(X_mat)

    # Cell-specific regression prediction
    m_ctrl = post * m_post + (1 - post) * m_pre
    r = Y - m_ctrl  # regression residuals

    # 3. Riesz representers (unnormalized)
    w_tp = treated * post                                      # treated, post
    w_tr = treated * (1 - post)                               # treated, pre
    w_cp = (1 - treated) * post        * ps / (1 - ps)       # ctrl,    post (IPW)
    w_cr = (1 - treated) * (1 - post)  * ps / (1 - ps)       # ctrl,    pre  (IPW)

    E_tp = w_tp.mean(); E_tr = w_tr.mean()
    E_cp = w_cp.mean(); E_cr = w_cr.mean()

    # 4. DR ATT = (att_trt_post - att_trt_pre) - (att_ctrl_post - att_ctrl_pre)
    att_tp = (w_tp * r).mean() / E_tp
    att_tr = (w_tr * r).mean() / E_tr
    att_cp = (w_cp * r).mean() / E_cp
    att_cr = (w_cr * r).mean() / E_cr
    att = (att_tp - att_tr) - (att_cp - att_cr)

    # 5. Influence function (uncentered; mean over n_k = att)
    psi = (w_tp / E_tp - w_tr / E_tr) * r - (w_tp / E_tp) * att_tp + (w_tr / E_tr) * att_tr \
        - (w_cp / E_cp - w_cr / E_cr) * r + (w_cp / E_cp) * att_cp - (w_cr / E_cr) * att_cr

    return att, psi

### 4.3  DR DDD assembler

In [23]:
def dr_ddd(df, outcome, impl_col, treat_col, post_col, X_cols, cluster_col):
    """
    Doubly-robust DDD estimator.

    DDD = ATT_k3 + ATT_k2 - ATT_k1, each a DR DiD (OVS 2025 eq. 4.1).

    Comparison groups vs treated (impl=1, treat=1):
      k3: (impl=1, treat=0)  within-country, between eligibility
      k2: (impl=0, treat=1)  between countries, among eligible
      k1: (impl=0, treat=0)  between countries, between eligibility

    SE from combined influence function, clustered by country.
    """
    T_mask = (df[impl_col] == 1) & (df[treat_col] == 1)
    comp = {
        'k3': (df[impl_col] == 1) & (df[treat_col] == 0),
        'k2': (df[impl_col] == 0) & (df[treat_col] == 1),
        'k1': (df[impl_col] == 0) & (df[treat_col] == 0),
    }

    n = len(df)
    post    = df[post_col].to_numpy().astype(float)
    Y       = df[outcome].to_numpy().astype(float)
    cluster = df[cluster_col].to_numpy()
    X_mat   = df[X_cols].to_numpy().astype(float)

    atts, psis_full = {}, {}
    for name, C_mask in comp.items():
        sel = (T_mask | C_mask).to_numpy()
        n_k = sel.sum()
        treated_flag = T_mask.to_numpy()[sel].astype(int)
        att_k, psi_k = dr_did_rc(
            Y[sel], post[sel], treated_flag, X_mat[sel], cluster[sel]
        )
        atts[name] = att_k
        # Extend psi to full n (zero for non-members), scale by n/n_k so
        # mean over full n equals att_k (OVS 2025 influence-function aggregation).
        psi_full = np.zeros(n)
        psi_full[sel] = psi_k * (n / n_k)
        psis_full[name] = psi_full

    ddd_att = atts['k3'] + atts['k2'] - atts['k1']

    # Combined DDD influence function
    psi_ddd = psis_full['k3'] + psis_full['k2'] - psis_full['k1']

    # Clustered SE (Cameron-Gelbach-Miller sandwich)
    cluster_vals = np.unique(cluster)
    G = len(cluster_vals)
    psi_c = psi_ddd - ddd_att  # centre around att for the sandwich
    cluster_sums = np.array([psi_c[cluster == c].sum() for c in cluster_vals])
    se = np.sqrt(G / (G - 1) * cluster_sums @ cluster_sums) / n

    t_stat = ddd_att / se
    p_val  = 2 * scipy_stats.norm.sf(abs(t_stat))

    return {
        'att': ddd_att, 'se': se, 't': t_stat, 'p': p_val,
        'ci_low':  ddd_att - 1.96 * se,
        'ci_high': ddd_att + 1.96 * se,
        'n': n, 'G': G,
        'components': {k: round(v, 6) for k, v in atts.items()},
        'psi': psi_ddd, 'cluster': cluster,
    }

### 4.4  Estimates for both outcomes

In [24]:
OUTCOMES_DR = {
    'target_engaged':    'Climate target engagement',
    'firm_growth_ord_d': 'Firm growth (binary)',
}

dr_results = {}
for col, label in OUTCOMES_DR.items():
    df_run = df_dr.dropna(subset=[col]).copy()
    result = dr_ddd(
        df_run,
        outcome     = col,
        impl_col    = 'implementation_country',
        treat_col   = 'treat',
        post_col    = 'post',
        X_cols      = X_cols_dr,
        cluster_col = 'ipscntry',
    )
    dr_results[col] = result
    print(f'\n{label}')
    print(f'  DR DDD ATT : {result["att"]:+.4f}')
    print(f'  SE (cluster): {result["se"]:.4f}')
    print(f'  t-stat      : {result["t"]:+.3f}')
    print(f'  p-value     : {result["p"]:.3f}')
    print(f'  95% CI      : [{result["ci_low"]:+.4f}, {result["ci_high"]:+.4f}]')
    print(f'  N obs       : {result["n"]:,}  |  G clusters: {result["G"]}')
    print(f'  Components  : k3={result["components"]["k3"]:+.4f}  ',
          f'k2={result["components"]["k2"]:+.4f}  ',
          f'k1={result["components"]["k1"]:+.4f}')


Climate target engagement
  DR DDD ATT : +0.1444
  SE (cluster): 0.0615
  t-stat      : +2.350
  p-value     : 0.019
  95% CI      : [+0.0240, +0.2649]
  N obs       : 28,236  |  G clusters: 27
  Components  : k3=+0.0413   k2=+0.1770   k1=+0.0738

Firm growth (binary)
  DR DDD ATT : -0.0324
  SE (cluster): 0.0581
  t-stat      : -0.558
  p-value     : 0.577
  95% CI      : [-0.1462, +0.0814]
  N obs       : 28,236  |  G clusters: 27
  Components  : k3=-0.0864   k2=-0.0313   k1=-0.0853


### 4.5  DR DDD vs 3WFE DDD — comparison

The 3WFE coefficients come from the baseline spec in Section 3 (`treat*post*impl`
+ sector + age FEs, analytic cluster-robust SE).

In [25]:
def stars(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''

# 3WFE DDD references from Section 3 (sector+age FE, analytic cluster-robust SE)
ref_3wfe = {
    'target_engaged':    (s3b.params['treat:post:impl'], s3b.pvalues['treat:post:impl']),
    'firm_growth_ord_d': (s3d.params['treat:post:impl'], s3d.pvalues['treat:post:impl']),
}

print('=' * 78)
print(f'{"Outcome":<22} {"Estimator":<20} {"ATT":>9} {"SE":>8} {"p":>8} {"CI 95%":>22}')
print('-' * 78)

for col, label in OUTCOMES_DR.items():
    dr  = dr_results[col]
    c3, p3 = ref_3wfe[col]
    se3 = s3b.bse['treat:post:impl'] if col == 'target_engaged' else s3d.bse['treat:post:impl']
    ci3_lo = c3 - 1.96 * se3;  ci3_hi = c3 + 1.96 * se3

    print(f'{label:<22} {"3WFE DDD (sec.3)":<20} {c3:>+9.4f} {se3:>8.4f}',
          f'{p3:>6.3f}{stars(p3):<2} [{ci3_lo:+.4f}, {ci3_hi:+.4f}]')
    print(f'{"":<22} {"DR DDD (sec.4)":<20} {dr["att"]:>+9.4f} {dr["se"]:>8.4f}',
          f'{dr["p"]:>6.3f}{stars(dr["p"]):<2} [{dr["ci_low"]:+.4f}, {dr["ci_high"]:+.4f}]')
    print()

print('=' * 78)
print('SE: both estimators cluster by ipscntry (27 country clusters).')
print('3WFE SE: analytic cluster-robust. DR DDD SE: influence-function cluster sandwich.')
print('3WFE relies on UNCONDITIONAL DDD-CPT; DR DDD is valid under CONDITIONAL DDD-CPT.')

Outcome                Estimator                  ATT       SE        p                 CI 95%
------------------------------------------------------------------------------
Climate target engagement 3WFE DDD (sec.3)       +0.1754   0.0507  0.001*** [+0.0761, +0.2747]
                       DR DDD (sec.6)         +0.1444   0.0615  0.019** [+0.0240, +0.2649]

Firm growth (binary)   3WFE DDD (sec.3)       -0.0524   0.0664  0.430   [-0.1826, +0.0777]
                       DR DDD (sec.6)         -0.0324   0.0581  0.577   [-0.1462, +0.0814]

SE: both estimators cluster by ipscntry (27 country clusters).
3WFE SE: analytic cluster-robust. DR DDD SE: influence-function cluster sandwich.
3WFE relies on UNCONDITIONAL DDD-CPT; DR DDD is valid under CONDITIONAL DDD-CPT.
Exposure: 3WFE uses original impl binary; DR DDD recodes Cyprus to impl=0.


### 4.6  Save DR DDD results

In [26]:
rows = []
for col, label in OUTCOMES_DR.items():
    r = dr_results[col]
    rows.append({
        'outcome': col, 'label': label,
        'att': r['att'], 'se': r['se'], 't': r['t'], 'p': r['p'],
        'ci_low': r['ci_low'], 'ci_high': r['ci_high'],
        'n': r['n'], 'G': r['G'],
        'att_k3': r['components']['k3'],
        'att_k2': r['components']['k2'],
        'att_k1': r['components']['k1'],
    })

dr_table = pd.DataFrame(rows)
dr_table.to_csv('dr_ddd_results.csv', index=False)
print('Saved: dr_ddd_results.csv')
print(dr_table.to_string(index=False))

Saved: dr_ddd_results.csv
          outcome                     label       att       se         t        p    ci_low  ci_high     n  G    att_k3    att_k2    att_k1
   target_engaged Climate target engagement  0.144445 0.061464  2.350063 0.018770  0.023975 0.264914 28236 27  0.041253  0.176981  0.073790
firm_growth_ord_d      Firm growth (binary) -0.032396 0.058053 -0.558043 0.576815 -0.146179 0.081387 28236 27 -0.086402 -0.031312 -0.085318


---
## 5. Overview Table

All parametric specifications (Sections 1–3) and the DR DDD estimates (Section 4).
Standard errors clustered by country (`ipscntry`).
Significance: \* p<0.1, \*\* p<0.05, \*\*\* p<0.01.

| Section | Models | Key coefficient |
|---|---|---|
| 1 OLS  | 1a–1d | `treat` |
| 2 DiD  | 2a–2d | `treat:post` |
| 3 DDD  | 3a–3d | `treat:post:impl` |
| 4 DR DDD | — | ATT (influence-function SE) |

In [ ]:
from statsmodels.iolib.summary2 import summary_col

# ── Parametric models 1a–3d ─────────────────────────────────────────────
comp_table = summary_col(
    [s1a, s1b, s1c, s1d, s2a, s2b, s2c, s2d, s3a, s3b, s3c, s3d],
    model_names=[
        "(1a)","(1b)","(1c)","(1d)",
        "(2a)","(2b)","(2c)","(2d)",
        "(3a)","(3b)","(3c)","(3d)",
    ],
    stars=True,
    float_format="%0.4f",
    regressor_order=["treat", "treat:post", "treat:post:impl"],
    drop_omitted=True,
    info_dict={
        "N":         lambda x: f"{int(x.nobs)}",
        "R-squared": lambda x: f"{x.rsquared:.3f}",
        "Outcome":   lambda x: x.model.endog_names,
    },
)
print(comp_table)

# ── DR DDD (Section 4) ──────────────────────────────────────────────────
def _stars(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''

print()
print('DR DDD estimates (Section 4)')
print('=' * 72)
print(f'{"Outcome":<30} {"ATT":>9} {"SE":>8} {"p":>8} {"95% CI":>22}')
print('-' * 72)
for col, label in OUTCOMES_DR.items():
    r = dr_results[col]
    print(f'{label:<30} {r["att"]:>+9.4f} {r["se"]:>8.4f} '
          f'{r["p"]:>6.3f}{_stars(r["p"]):<2} [{r["ci_low"]:+.4f}, {r["ci_high"]:+.4f}]')
print('=' * 72)
print(f'N: {dr_results["target_engaged"]["n"]:,}  |  G clusters: {dr_results["target_engaged"]["G"]}')
print('SE: influence-function cluster sandwich (ipscntry).')

In [ ]:
with open('overview_table.txt', 'w', encoding='utf-8') as f:
    f.write(str(comp_table))
    f.write('\n\nDR DDD estimates (Section 4)\n')
    f.write('=' * 72 + '\n')
    f.write(f'{"Outcome":<30} {"ATT":>9} {"SE":>8} {"p":>8} {"95% CI":>22}\n')
    f.write('-' * 72 + '\n')
    for col, label in OUTCOMES_DR.items():
        r = dr_results[col]
        f.write(f'{label:<30} {r["att"]:>+9.4f} {r["se"]:>8.4f} '
                f'{r["p"]:>6.3f}{_stars(r["p"]):<2} [{r["ci_low"]:+.4f}, {r["ci_high"]:+.4f}]\n')
    f.write('=' * 72 + '\n')
print('Saved: overview_table.txt')

---
## 6. Robustness Checks

| Subsection | Check |
|---|---|
| 6.1 | Logistic regression for binary growth outcome |
| 6.2 | Repeated cross-section composition checks |
| 6.3 | Placebo treatment definitions |
| 6.4 | DDD first stage and covariate overlap |
| 6.5 | DiD with alternative outcome variables |
| 6.6 | OLS / DiD / DDD with East\u2013West control |

### 6.1  Logistic Regression for Binary Growth Outcome

`firm_growth_ord_d` is binary (0/1). The OLS linear probability model (LPM) used in
Sections 1\u20133 is consistent but may predict probabilities outside [0, 1].  These
logit models replicate the OLS, DiD, and DDD structures.
Coefficients are log-odds; the adjacent LPM column (s1b/s2b/s3d) enables comparison.

In [ ]:
# 6.1  Logit models for firm_growth_ord_d
lg_ols = smf.logit('firm_growth_ord_d ~ treat + C(nace_b) + C(firm_age_ord)',
                    data=df_s).fit(**cl(df_s))
lg_did = smf.logit('firm_growth_ord_d ~ treat*post + C(nace_b) + C(firm_age_ord)',
                    data=df_s).fit(**cl(df_s))
lg_ddd = smf.logit('firm_growth_ord_d ~ treat*post*impl + C(nace_b) + C(firm_age_ord)',
                    data=df_d).fit(**cl(df_d))

logit_specs = [
    ('OLS analog', lg_ols, 'treat',           s1b, 'treat'),
    ('DiD',        lg_did, 'treat:post',       s2b, 'treat:post'),
    ('DDD',        lg_ddd, 'treat:post:impl',  s3d, 'treat:post:impl'),
]
print('%-14s %11s %8s %8s   %11s %8s %8s' % (
      'Model', 'Logit coef', 'SE', 'p', 'LPM coef', 'SE', 'p'))
print('-' * 76)
for lbl, ml, kl, mo, ko in logit_specs:
    cl_ = ml.params[kl]; sl_ = ml.bse[kl]; pl_ = ml.pvalues[kl]
    co_ = mo.params[ko]; so_ = mo.bse[ko]; po_ = mo.pvalues[ko]
    sg  = '***' if pl_<0.01 else '**' if pl_<0.05 else '*' if pl_<0.10 else ''
    print('%-14s %+11.4f %8.4f %6.3f%-3s  %+11.4f %8.4f %6.3f' % (
          lbl, cl_, sl_, pl_, sg, co_, so_, po_))
print()
print('Logit coefs = log-odds. LPM coefs from s1b/s2b/s3d. SE clustered by ipscntry.')

### 6.2  Repeated Cross-Section Composition Checks

The survey is not a true firm panel: each wave samples independently.
Observed outcome changes could reflect shifts in sample composition (sector mix,
size distribution, geography) rather than genuine firm-level responses.
We test whether key observable characteristics differ across the 2022 and 2024 waves.

In [ ]:
# 6.2  Composition checks
from scipy.stats import chi2_contingency

print('Chi-square tests: independence of composition variable and survey year (df_s)')
print('%-30s %8s %4s %8s' % ('Variable', 'chi2', 'df', 'p-value'))
print('-' * 55)
for col in ['treat', 'nace_b', 'east_west', 'firm_age_ord']:
    sub = df_s[['year', col]].dropna()
    ch, pv, dof, _ = chi2_contingency(pd.crosstab(sub['year'], sub[col]))
    sg = '***' if pv<0.01 else '**' if pv<0.05 else '*' if pv<0.10 else ''
    print('%-30s %8.2f %4d %8.3f%s' % (col, ch, dof, pv, sg))

print()
print('Mean outcomes by year x treat group (df_s)')
tbl = (df_s.groupby(['year', 'treat'])
           [['target_engaged', 'firm_growth_ord_d', 'firm_age_ord']]
           .mean())
print(tbl.to_string(float_format='%.4f'))

print()
print('Sample size by year (df_s)')
print(df_s.groupby('year').size().rename('n').to_string())

### 6.3  Placebo Treatment Definitions

The baseline `treat` indicator is 1 when a firm meets **either** the employee
(\u2265 250) **or** turnover (\u2265 \u20ac10 M) threshold (CSRD Article 3).
We re-estimate the DiD and DDD using three alternative definitions built from the
same pre-computed binary indicators in the parquet:

| Definition | Rule |
|---|---|
| Baseline (any) | employee \u2265 250 **OR** turnover \u2265 \u20ac10 M (main spec) |
| Employees only | `employee_treatment = 1` regardless of turnover |
| Turnover only  | `turnover_treatment = 1` regardless of employees |
| Both (strict)  | `employee_treatment = 1` **AND** `turnover_treatment = 1` |

In [ ]:
# 6.3  Placebo treatment definitions
df_s_pl = df_s.copy()
df_d_pl = df_d.copy()
for df_ in [df_s_pl, df_d_pl]:
    df_['treat_emp']    = df_['employee_treatment'].astype(float)
    df_['treat_turn']   = df_['turnover_treatment'].astype(float)
    df_['treat_strict'] = (df_['employee_treatment'] * df_['turnover_treatment']).astype(float)

print('Treatment group size by definition (df_s)')
for tcol, lbl in [('treat','Baseline'),('treat_emp','Employees only'),
                   ('treat_turn','Turnover only'),('treat_strict','Both (strict)')]:
    n = int(df_s_pl[tcol].fillna(0).sum())
    print('  %-18s treat=1: %5d  (%.1f%%)' % (lbl, n, 100*n/len(df_s_pl)))

print()
print('%-20s %10s %8s %8s   %10s %8s %8s' % (
      'Treatment def.', 'DiD coef', 'SE', 'p', 'DDD coef', 'SE', 'p'))
print('-' * 80)
for label, tcol in [('Baseline (any)',  'treat'),
                     ('Employees only', 'treat_emp'),
                     ('Turnover only',  'treat_turn'),
                     ('Both (strict)',  'treat_strict')]:
    if tcol == 'treat':
        md_ = s2a;  m3_ = s3b
    else:
        ss = df_s_pl.dropna(subset=[tcol])
        sd = df_d_pl.dropna(subset=[tcol])
        md_ = smf.ols('target_engaged ~ %s*post + C(nace_b) + C(firm_age_ord)' % tcol,
                       data=ss).fit(cov_type='cluster', cov_kwds={'groups': ss['ipscntry']})
        m3_ = smf.ols('target_engaged ~ %s*post*impl + C(nace_b) + C(firm_age_ord)' % tcol,
                       data=sd).fit(cov_type='cluster', cov_kwds={'groups': sd['ipscntry']})
    kd = tcol + ':post';  k3 = tcol + ':post:impl'
    cd = md_.params[kd]; sd_ = md_.bse[kd]; pd_ = md_.pvalues[kd]
    c3 = m3_.params[k3]; s3_ = m3_.bse[k3]; p3_ = m3_.pvalues[k3]
    sgd = '***' if pd_<0.01 else '**' if pd_<0.05 else '*' if pd_<0.10 else ''
    sg3 = '***' if p3_<0.01 else '**' if p3_<0.05 else '*' if p3_<0.10 else ''
    print('%-20s %+10.4f %8.4f %6.3f%-3s  %+10.4f %8.4f %6.3f%s' % (
          label, cd, sd_, pd_, sgd, c3, s3_, p3_, sg3))

### 6.4  DDD First Stage and Covariate Overlap

**Panel A** plots mean `target_engaged` by year for the four DDD cells (treat \u00d7 impl).
Non-parallel pre-trends between the two treatment-group pairs would undermine the
identification assumption.

**Panel B** shows the `firm_age_ord` distribution within each cell as a covariate
overlap check.  The DR DDD estimator requires common support across all four groups.

The group count table below confirms adequate observations in each cell.

In [ ]:
# 6.4  DDD first stage and overlap
import matplotlib.ticker as mticker

print('Observation counts by (impl x treat x post)')
cnt = df_d.groupby(['impl', 'treat', 'post']).size().reset_index(name='n')
print(cnt.to_string(index=False))
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: mean outcome trajectories for 4 groups
style_map = {(1, 1): ('steelblue', '-',  'o'),
             (1, 0): ('steelblue', '--', 's'),
             (0, 1): ('tomato',    '-',  'o'),
             (0, 0): ('tomato',    '--', 's')}
for (impl_v, treat_v), (col, ls, mk) in style_map.items():
    sub = df_d[(df_d['impl'] == impl_v) & (df_d['treat'] == treat_v)]
    means = sub.groupby('year')['target_engaged'].mean()
    axes[0].plot(means.index, means.values, color=col, linestyle=ls,
                 marker=mk, linewidth=1.8,
                 label='impl=%d, treat=%d' % (impl_v, treat_v))
axes[0].set_title('Target engagement by group and year', fontsize=10)
axes[0].set_xlabel('Survey year')
axes[0].set_ylabel('Mean target_engaged')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Panel B: firm_age_ord distribution — overlap check
color_b = {(1, 1): 'steelblue', (1, 0): 'lightblue',
           (0, 1): 'tomato',    (0, 0): 'lightsalmon'}
for (impl_v, treat_v), col in color_b.items():
    vals = df_d[(df_d['impl'] == impl_v) & (df_d['treat'] == treat_v)]['firm_age_ord'].dropna()
    axes[1].hist(vals, bins=[0.5, 1.5, 2.5, 3.5, 4.5], alpha=0.55, density=True,
                 color=col, label='impl=%d, treat=%d (n=%d)' % (impl_v, treat_v, len(vals)))
axes[1].set_title('Firm age distribution by group (overlap)', fontsize=10)
axes[1].set_xlabel('firm_age_ord  (1=youngest, 4=oldest)')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3)
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.tight_layout()
plt.savefig('ddd_overlap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ddd_overlap.png')

### 6.5  DiD with Alternative Outcome Variables

Running DiD on pre-determined firm characteristics (`firm_age_ord`) serves as a
**falsification test**: a significant `treat:post` coefficient would indicate
compositional shifts rather than genuine treatment effects.
For forward-looking variables (`green_offer_ord_d`, `green_staff_any`, etc.) the DiD
captures potential spillover mechanisms of the CSRD.

In [ ]:
# 6.5  DiD on alternative outcomes
alt_out = [
    ('firm_age_ord',            'Firm age ordinal [falsification]'),
    ('green_staff_any',         'Green staff (any)'),
    ('investment_dummy',        'Investment dummy'),
    ('barrier_institutional_d', 'Institutional barriers'),
    ('support_knowledge_d',     'Knowledge support'),
    ('green_offer_ord_d',       'Green offer (binary)'),
]

print('%-40s %10s %8s %8s  %7s' % ('Outcome', 'treat:post', 'SE', 'p', 'N'))
print('-' * 80)
for col, label in alt_out:
    sub = df_s.dropna(subset=[col]).copy()
    # firm_age_ord cannot appear as both LHS and RHS control
    rhs = ('treat*post + C(nace_b)' if col == 'firm_age_ord'
           else 'treat*post + C(nace_b) + C(firm_age_ord)')
    m = smf.ols('%s ~ %s' % (col, rhs), data=sub).fit(
            cov_type='cluster', cov_kwds={'groups': sub['ipscntry']})
    c = m.params.get('treat:post', float('nan'))
    s = m.bse.get('treat:post',   float('nan'))
    p = m.pvalues.get('treat:post', float('nan'))
    sg = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else ''
    print('%-40s %+10.4f %8.4f %6.3f%-3s %7d' % (label, c, s, p, sg, int(m.nobs)))
print()
print('DiD: treat*post + sector FEs + firm_age FE (omitted when outcome = firm_age_ord).')
print('SE clustered by ipscntry. Significant firm_age_ord coef = composition shift.')

### 6.6  OLS / DiD / DDD with East\u2013West Control

`east_west = 1` for Western EU member states, `0` for Central/Eastern EU.
Adding this country-level dummy controls for unobserved geographic differences
in baseline climate engagement.  In DDD models `east_west` is collinear with
country-level variation in `impl`, so coefficient SEs may be inflated; the
direction and significance of `treat:post:impl` provides the relevant comparison.

| New model | Base | Added |
|---|---|---|
| s1a\_ew / s1b\_ew | s1a / s1b | + `east_west` (OLS) |
| s2a\_ew / s2b\_ew | s2a / s2b | + `east_west` (DiD) |
| s3b\_ew / s3d\_ew | s3b / s3d | + `east_west` (DDD) |

In [ ]:
# 6.6  East-West robustness
s1a_ew = smf.ols('target_engaged    ~ treat + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_s).fit(**cl(df_s))
s1b_ew = smf.ols('firm_growth_ord_d ~ treat + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_s).fit(**cl(df_s))
s2a_ew = smf.ols('target_engaged    ~ treat*post + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_s).fit(**cl(df_s))
s2b_ew = smf.ols('firm_growth_ord_d ~ treat*post + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_s).fit(**cl(df_s))
s3b_ew = smf.ols('target_engaged    ~ treat*post*impl + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_d).fit(**cl(df_d))
s3d_ew = smf.ols('firm_growth_ord_d ~ treat*post*impl + east_west + C(nace_b) + C(firm_age_ord)',
                  data=df_d).fit(**cl(df_d))

ew_specs = [
    ('OLS 1a', s1a_ew, s1a, 'treat'),
    ('OLS 1b', s1b_ew, s1b, 'treat'),
    ('DiD 2a', s2a_ew, s2a, 'treat:post'),
    ('DiD 2b', s2b_ew, s2b, 'treat:post'),
    ('DDD 3b', s3b_ew, s3b, 'treat:post:impl'),
    ('DDD 3d', s3d_ew, s3d, 'treat:post:impl'),
]

print('%-10s %11s %8s %6s   %11s %8s %6s   %9s %8s' % (
      'Model', '+EW coef', 'SE', 'p', 'Base coef', 'SE', 'p', 'EW coef', 'EW SE'))
print('-' * 88)
for lbl, mew, mbase, key in ew_specs:
    c_ew = mew.params[key];   s_ew = mew.bse[key];   p_ew = mew.pvalues[key]
    c_b  = mbase.params[key]; s_b  = mbase.bse[key]; p_b  = mbase.pvalues[key]
    c_e  = mew.params.get('east_west', float('nan'))
    s_e  = mew.bse.get('east_west',    float('nan'))
    sg   = '***' if p_ew<0.01 else '**' if p_ew<0.05 else '*' if p_ew<0.10 else ''
    print('%-10s %+11.4f %8.4f %4.3f%-3s  %+11.4f %8.4f %4.3f   %+9.4f %8.4f' % (
          lbl, c_ew, s_ew, p_ew, sg, c_b, s_b, p_b, c_e, s_e))